# 🛡️ AI-Based Intrusion Detection & Automated Response System (IDRS)
## For Educational Web Platforms — Tunisia Context
---
## PART 4: Anomaly Detection · Zero-Day Simulation · Online Learning

**Depends on:** Parts 1–3 outputs (`X_train_res.npy`, `X_test_scaled.npy`, `deep_model_registry.json`)

**Scope of Part 4:**
- ✅ **Autoencoder** — unsupervised reconstruction-error anomaly scoring
- ✅ **Isolation Forest** — tree-based density-free outlier detection
- ✅ **One-Class SVM** — kernel-based boundary learning on normal traffic
- ✅ **Ensemble anomaly scorer** — weighted combination of all three signals
- ✅ **Zero-day simulation** — novel attack family holdout, detection rate analysis
- ✅ **BETH Dataset** integration note + host-behavioral feature processing
- ✅ **River ADWIN** — online learning with concept drift detection
- ✅ Full ROC · PR · threshold sweep evaluation for every detector
- ✅ All models serialised for Part 6 response engine

---
## ⚙️ CELL 1 — Imports & Config

In [ ]:
# ============================================================
# IDRS Part 4 — Imports & Configuration
# ============================================================
import os, sys, json, warnings, logging, time
from pathlib import Path
from typing  import Dict, List, Tuple, Optional, Any

import numpy  as np
import pandas as pd
import scipy.stats as sp_stats
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
import seaborn as sns

# ── Scikit-learn ──────────────────────────────────────────
from sklearn.ensemble          import IsolationForest
from sklearn.svm               import OneClassSVM
from sklearn.preprocessing     import StandardScaler
from sklearn.metrics           import (
    roc_auc_score, average_precision_score, roc_curve,
    precision_recall_curve, f1_score, confusion_matrix,
    classification_report, accuracy_score
)
from sklearn.model_selection   import train_test_split
import joblib

# ── PyTorch ───────────────────────────────────────────────
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data  import DataLoader, TensorDataset
from torch.optim       import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.cuda.amp    import GradScaler, autocast

# ── Online learning / concept drift ──────────────────────
from river import drift, stream, metrics as river_metrics
from river import linear_model, preprocessing as river_prep
from river import ensemble as river_ensemble

from tqdm import tqdm
warnings.filterwarnings('ignore')

# ── Reproducibility ───────────────────────────────────────
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE  = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP = torch.cuda.is_available()

# ── Directories ───────────────────────────────────────────
BASE_DIR   = Path('.')
OUTPUT_DIR = BASE_DIR / 'outputs'
MODEL_DIR  = BASE_DIR / 'models'
LOG_DIR    = BASE_DIR / 'logs'
for d in [OUTPUT_DIR, MODEL_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)s | %(message)s',
    handlers=[
        logging.FileHandler(LOG_DIR / 'idrs_part4.log'),
        logging.StreamHandler(sys.stdout)
    ]
)
logger = logging.getLogger('IDRS-P4')

plt.rcParams.update({
    'figure.facecolor':'white','axes.facecolor':'#f8f9fa',
    'axes.grid':True,'grid.alpha':0.4,'font.size':11,
    'axes.titlesize':13,'figure.dpi':120,
})
PALETTE = {
    'BENIGN':'#2ecc71','DoS':'#e74c3c','DDoS':'#c0392b',
    'Probe':'#f39c12','WebAttack':'#9b59b6','BruteForce':'#3498db',
    'Botnet':'#1abc9c','Exploit':'#e67e22','UNKNOWN':'#95a5a6'
}
logger.info('Part 4 environment ready. Device=%s', DEVICE)
print(f'✅ Part 4 ready | Device: {DEVICE}')

---
## 📥 CELL 2 — Load Data & Construct Anomaly Detection Splits

In [ ]:
# ============================================================
# IDRS Part 4 — Data Loading
#
# Anomaly detection uses a DIFFERENT split strategy:
#   Train  : BENIGN only      → teach model what "normal" looks like
#   Test   : BENIGN + ATTACKS → evaluate detection capability
#
# This mirrors real-world IDS deployment where you train on
# clean traffic logs and must detect unknown deviations.
# ============================================================

X_train_all = np.load(OUTPUT_DIR / 'X_train_res.npy').astype(np.float32)
y_train_all = np.load(OUTPUT_DIR / 'y_train_res.npy').astype(np.int64)
X_test_all  = np.load(OUTPUT_DIR / 'X_test_scaled.npy').astype(np.float32)
y_test_all  = np.load(OUTPUT_DIR / 'y_test.npy').astype(np.int64)
X_val_all   = np.load(OUTPUT_DIR / 'X_val_scaled.npy').astype(np.float32)
y_val_all   = np.load(OUTPUT_DIR / 'y_val.npy').astype(np.int64)

with open(OUTPUT_DIR / 'label_classes.json') as f:
    lc = json.load(f)
INT_TO_CLASS = {int(k): v for k, v in lc.items()}
CLASS_NAMES  = [INT_TO_CLASS[i] for i in sorted(INT_TO_CLASS)]
N_CLASSES    = len(CLASS_NAMES)
BENIGN_ID    = next((i for i,c in INT_TO_CLASS.items() if c=='BENIGN'), 0)

with open(MODEL_DIR / 'preprocessor_meta.json') as f:
    prep = json.load(f)
FEATURE_COLS = prep['feature_cols']
N_FEATURES   = len(FEATURE_COLS)

# ── Anomaly detection split ───────────────────────────────
# Training: BENIGN only (from original unbalanced train set)
# We use the RAW (pre-SMOTE) distribution so benign is dominant
benign_mask_tr  = (y_train_all == BENIGN_ID)
X_ad_train      = X_train_all[benign_mask_tr]   # normal-only train

# Test: full mix of benign + attacks
X_ad_test  = X_test_all
y_ad_test  = (y_test_all != BENIGN_ID).astype(int)  # binary: 0=normal, 1=anomaly

# Val: for threshold tuning
X_ad_val   = X_val_all
y_ad_val   = (y_val_all  != BENIGN_ID).astype(int)

print(f'✅ Anomaly detection splits:')
print(f'   AD-Train (BENIGN only) : {X_ad_train.shape}')
print(f'   AD-Val                 : {X_ad_val.shape}  '
      f'(anomaly rate: {y_ad_val.mean()*100:.1f}%)')
print(f'   AD-Test                : {X_ad_test.shape}  '
      f'(anomaly rate: {y_ad_test.mean()*100:.1f}%)')
print(f'   BENIGN_ID={BENIGN_ID}  |  N_FEATURES={N_FEATURES}')

---
## 🔬 CELL 3 — Anomaly Evaluation Toolkit

In [ ]:
# ============================================================
# IDRS Part 4 — Anomaly Detector Evaluation Toolkit
# ============================================================

def evaluate_anomaly_detector(
    scores      : np.ndarray,   # higher score = more anomalous
    y_true      : np.ndarray,   # binary: 0=normal, 1=anomaly
    detector_name: str,
    split_name  : str = 'Test',
    save_prefix : str = '',
    plot        : bool = True,
) -> Dict[str, Any]:
    """
    Evaluate a binary anomaly score vector.
    Finds optimal threshold via Youden's J statistic on the given split.
    """
    # ── ROC & PR ──────────────────────────────────────────
    fpr, tpr, roc_thresh = roc_curve(y_true, scores)
    roc_auc = roc_auc_score(y_true, scores)

    prec, rec, pr_thresh = precision_recall_curve(y_true, scores)
    pr_auc = average_precision_score(y_true, scores)

    # ── Optimal threshold: Youden's J = TPR - FPR ────────
    j_scores = tpr - fpr
    best_idx = int(np.argmax(j_scores))
    opt_thresh = float(roc_thresh[best_idx])

    y_pred    = (scores >= opt_thresh).astype(int)
    acc       = accuracy_score(y_true, y_pred)
    f1        = f1_score(y_true, y_pred, zero_division=0)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred,
                                       labels=[0,1]).ravel()
    tpr_val   = tp / (tp + fn + 1e-9)
    fpr_val   = fp / (fp + tn + 1e-9)
    fnr_val   = fn / (fn + tp + 1e-9)   # miss rate

    results = {
        'detector'   : detector_name,
        'split'      : split_name,
        'roc_auc'    : round(roc_auc,  4),
        'pr_auc'     : round(pr_auc,   4),
        'accuracy'   : round(acc,      4),
        'f1'         : round(f1,       4),
        'tpr'        : round(tpr_val,  4),
        'fpr'        : round(fpr_val,  4),
        'fnr'        : round(fnr_val,  4),  # miss rate — critical for IDS
        'opt_thresh' : round(opt_thresh, 6),
        'tp': int(tp), 'fp': int(fp),
        'tn': int(tn), 'fn': int(fn),
    }

    print(f'\n{"─"*56}')
    print(f'  {detector_name} — {split_name}')
    print(f'{"─"*56}')
    print(f'  ROC-AUC  : {roc_auc:.4f}   PR-AUC  : {pr_auc:.4f}')
    print(f'  F1       : {f1:.4f}   Accuracy: {acc:.4f}')
    print(f'  TPR      : {tpr_val:.4f}   FPR     : {fpr_val:.4f}')
    print(f'  Miss rate: {fnr_val:.4f}   Threshold: {opt_thresh:.4f}')
    print(f'  TP={tp}  FP={fp}  TN={tn}  FN={fn}')

    if not plot:
        return results

    fig, axes = plt.subplots(1, 3, figsize=(19, 5))

    # ROC
    ax = axes[0]
    ax.plot(fpr, tpr, lw=2.5, color='#3498db',
            label=f'ROC AUC={roc_auc:.4f}')
    ax.fill_between(fpr, tpr, alpha=0.12, color='#3498db')
    ax.plot([0,1],[0,1],'k--',lw=1,alpha=0.5)
    ax.scatter(fpr[best_idx], tpr[best_idx], s=120, zorder=5,
               color='#e74c3c', label=f'Optimal t={opt_thresh:.3f}')
    ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
    ax.set_title(f'ROC Curve\n{detector_name}', fontweight='bold')
    ax.legend(fontsize=9)

    # PR
    ax = axes[1]
    ax.plot(rec, prec, lw=2.5, color='#9b59b6',
            label=f'PR AUC={pr_auc:.4f}')
    ax.fill_between(rec, prec, alpha=0.12, color='#9b59b6')
    base = y_true.mean()
    ax.axhline(base, color='k', ls='--', lw=1, alpha=0.5,
               label=f'Baseline={base:.3f}')
    ax.set_xlabel('Recall'); ax.set_ylabel('Precision')
    ax.set_title(f'PR Curve\n{detector_name}', fontweight='bold')
    ax.legend(fontsize=9)

    # Score distribution
    ax = axes[2]
    benign_scores = scores[y_true == 0]
    attack_scores = scores[y_true == 1]
    ax.hist(benign_scores, bins=80, alpha=0.6, color='#2ecc71',
            label='Benign', density=True, edgecolor='none')
    ax.hist(attack_scores, bins=80, alpha=0.6, color='#e74c3c',
            label='Attack', density=True, edgecolor='none')
    ax.axvline(opt_thresh, color='black', lw=2, ls='--',
               label=f'Threshold={opt_thresh:.3f}')
    ax.set_xlabel('Anomaly Score')
    ax.set_ylabel('Density')
    ax.set_title(f'Score Distribution\n{detector_name}', fontweight='bold')
    ax.legend(fontsize=9)

    plt.suptitle(
        f'IDRS Anomaly Detector — {detector_name} | '
        f'AUC={roc_auc:.4f} · F1={f1:.4f} · Miss={fnr_val:.4f}',
        fontsize=11, fontweight='bold'
    )
    plt.tight_layout()
    fname = OUTPUT_DIR / f'anomaly_{save_prefix}_{split_name.lower()}.png'
    plt.savefig(fname, bbox_inches='tight', dpi=150)
    plt.show()
    print(f'💾 Saved: {fname.name}')

    return results


print('✅ Anomaly evaluation toolkit ready.')

---
## 🧠 CELL 4 — Deep Autoencoder Architecture

In [ ]:
# ============================================================
# IDRS Part 4 — Deep Autoencoder for Anomaly Detection
#
# Principle:
#   Trained ONLY on benign traffic → learns to reconstruct
#   normal patterns with low error. Attack traffic has
#   unfamiliar structure → high reconstruction error = anomaly.
#
# Architecture: Symmetric encoder-decoder with skip connections
#   Encoder: N_FEAT → 256 → 128 → 64 → LATENT(32)
#   Decoder: 32 → 64 → 128 → 256 → N_FEAT
#   Skip connections: encoder[i] → decoder[n-i] (U-Net style)
#   This prevents information bottleneck from losing fine details
#   needed to accurately reconstruct benign features.
# ============================================================

class EncoderBlock(nn.Module):
    def __init__(self, in_dim, out_dim, dropout=0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, out_dim),
            nn.BatchNorm1d(out_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )
    def forward(self, x):
        return self.net(x)


class DecoderBlock(nn.Module):
    def __init__(self, in_dim, out_dim, dropout=0.2, last=False):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, out_dim),
            nn.Sigmoid() if last else nn.Sequential(
                nn.BatchNorm1d(out_dim),
                nn.GELU(),
                nn.Dropout(dropout),
            )
        )
    def forward(self, x):
        return self.net(x)


class IDRSAutoencoder(nn.Module):
    """
    Deep Autoencoder with skip connections for anomaly detection.
    Anomaly score = mean squared reconstruction error per sample.
    """
    def __init__(self, n_features: int, latent_dim: int = 32,
                 dropout: float = 0.2):
        super().__init__()
        self.n_features = n_features
        self.latent_dim = latent_dim

        # ── Encoder ───────────────────────────────────────
        self.enc1 = EncoderBlock(n_features, 256, dropout)
        self.enc2 = EncoderBlock(256,        128, dropout)
        self.enc3 = EncoderBlock(128,         64, dropout)
        self.enc4 = nn.Sequential(
            nn.Linear(64, latent_dim),
            nn.Tanh(),
        )

        # ── Decoder (skip connections double input dims) ──
        self.dec4 = DecoderBlock(latent_dim,      64,         dropout)
        self.dec3 = DecoderBlock(64  + 64,        128,        dropout)  # skip from enc3
        self.dec2 = DecoderBlock(128 + 128,       256,        dropout)  # skip from enc2
        self.dec1 = DecoderBlock(256 + 256,       n_features, dropout,  # skip from enc1
                                 last=False)
        self.out  = nn.Linear(n_features, n_features)

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def encode(self, x: torch.Tensor) -> Tuple[torch.Tensor, ...]:
        h1 = self.enc1(x)
        h2 = self.enc2(h1)
        h3 = self.enc3(h2)
        z  = self.enc4(h3)
        return z, h1, h2, h3

    def decode(self, z, h1, h2, h3) -> torch.Tensor:
        d4 = self.dec4(z)
        d3 = self.dec3(torch.cat([d4, h3], dim=-1))
        d2 = self.dec2(torch.cat([d3, h2], dim=-1))
        d1 = self.dec1(torch.cat([d2, h1], dim=-1))
        return self.out(d1)

    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        z, h1, h2, h3 = self.encode(x)
        x_hat = self.decode(z, h1, h2, h3)
        return x_hat, z

    @torch.no_grad()
    def anomaly_score(self, x: torch.Tensor) -> np.ndarray:
        """Per-sample mean squared reconstruction error."""
        self.eval()
        x_hat, _ = self(x)
        mse = ((x - x_hat) ** 2).mean(dim=-1)
        return mse.cpu().numpy()


# ── Test forward pass ─────────────────────────────────────
ae = IDRSAutoencoder(N_FEATURES, latent_dim=32)
dummy = torch.randn(4, N_FEATURES)
xhat, z = ae(dummy)
print(f'✅ IDRSAutoencoder:')
print(f'   Input     : {dummy.shape}')
print(f'   Latent z  : {z.shape}')
print(f'   Recon x_hat: {xhat.shape}')
print(f'   Params    : {sum(p.numel() for p in ae.parameters()):,}')
del dummy, xhat, z

---
## 🚂 CELL 5 — Train Autoencoder on Benign Traffic

In [ ]:
# ============================================================
# IDRS Part 4 — Autoencoder Training
# ============================================================

AE_EPOCHS    = 80
AE_LR        = 1e-3
AE_BATCH     = 512
AE_PATIENCE  = 15
AE_LATENT    = 32

# ── DataLoaders (benign-only train) ──────────────────────
X_ae_tr = torch.tensor(X_ad_train, dtype=torch.float32)
X_ae_vl = torch.tensor(X_ad_val,   dtype=torch.float32)

ae_train_ds = TensorDataset(X_ae_tr, X_ae_tr)   # input = target
ae_val_ds   = TensorDataset(X_ae_vl, X_ae_vl)

ae_train_loader = DataLoader(ae_train_ds, batch_size=AE_BATCH,
                              shuffle=True,  drop_last=True)
ae_val_loader   = DataLoader(ae_val_ds,   batch_size=AE_BATCH*2,
                              shuffle=False)

# ── Model & optimiser ─────────────────────────────────────
ae_model   = IDRSAutoencoder(N_FEATURES, latent_dim=AE_LATENT, dropout=0.15).to(DEVICE)
ae_optim   = AdamW(ae_model.parameters(), lr=AE_LR, weight_decay=1e-4)
ae_sched   = CosineAnnealingLR(ae_optim, T_max=AE_EPOCHS, eta_min=AE_LR*0.01)
ae_scaler  = GradScaler(enabled=USE_AMP)
ae_crit    = nn.MSELoss(reduction='mean')

ae_history = {'train_loss': [], 'val_loss': [], 'lr': []}
best_val   = np.inf
patience_c = 0
best_state = None

print(f'🧠 Training Autoencoder on BENIGN traffic only...')
print(f'   Train samples: {len(X_ae_tr):,}  |  '
      f'Val samples: {len(X_ae_vl):,}  |  Epochs: {AE_EPOCHS}')

for epoch in range(1, AE_EPOCHS + 1):
    # ── Train ──────────────────────────────────────────────
    ae_model.train()
    tr_loss = 0.0
    for xb, _ in ae_train_loader:
        xb = xb.to(DEVICE)
        ae_optim.zero_grad(set_to_none=True)
        with autocast(enabled=USE_AMP):
            xhat, z = ae_model(xb)
            loss = ae_crit(xhat, xb)
        ae_scaler.scale(loss).backward()
        ae_scaler.unscale_(ae_optim)
        nn.utils.clip_grad_norm_(ae_model.parameters(), 1.0)
        ae_scaler.step(ae_optim)
        ae_scaler.update()
        tr_loss += loss.item() * len(xb)
    tr_loss /= len(ae_train_ds)

    # ── Validate ───────────────────────────────────────────
    ae_model.eval()
    vl_loss = 0.0
    with torch.no_grad():
        for xb, _ in ae_val_loader:
            xb = xb.to(DEVICE)
            xhat, _ = ae_model(xb)
            vl_loss += ae_crit(xhat, xb).item() * len(xb)
    vl_loss /= len(ae_val_ds)

    ae_sched.step()
    lr = ae_optim.param_groups[0]['lr']
    ae_history['train_loss'].append(tr_loss)
    ae_history['val_loss'].append(vl_loss)
    ae_history['lr'].append(lr)

    improved = '✅' if vl_loss < best_val else '  '
    if epoch % 10 == 0 or epoch <= 5:
        print(f'  Ep {epoch:03d}/{AE_EPOCHS} {improved} '
              f'TrLoss={tr_loss:.6f}  ValLoss={vl_loss:.6f}  '
              f'LR={lr:.2e}')

    if vl_loss < best_val:
        best_val   = vl_loss
        patience_c = 0
        best_state = {k: v.clone() for k, v in ae_model.state_dict().items()}
    else:
        patience_c += 1
        if patience_c >= AE_PATIENCE:
            print(f'  ⏹️  Early stopping at epoch {epoch}  (best val={best_val:.6f})')
            break

ae_model.load_state_dict(best_state)
print(f'\n  ✅ Best val MSE: {best_val:.6f}')

# ── Plot training loss ─────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
epochs_range = range(1, len(ae_history['train_loss'])+1)

ax = axes[0]
ax.plot(epochs_range, ae_history['train_loss'], lw=2, color='#3498db', label='Train MSE')
ax.plot(epochs_range, ae_history['val_loss'],   lw=2, color='#e74c3c',
        ls='--', label='Val MSE')
ax.set_yscale('log')
ax.set_title('Autoencoder Training Loss (log)', fontweight='bold')
ax.set_xlabel('Epoch'); ax.set_ylabel('MSE (log)')
ax.legend()

ax = axes[1]
ax.plot(epochs_range, ae_history['lr'], lw=2, color='#2ecc71')
ax.set_title('Learning Rate Schedule', fontweight='bold')
ax.set_xlabel('Epoch'); ax.set_ylabel('LR')

plt.suptitle('Autoencoder Training — Benign Traffic Only', fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'ae_training.png', bbox_inches='tight', dpi=150)
plt.show()

# Save
torch.save({'model_state': ae_model.state_dict(),
            'n_features': N_FEATURES, 'latent_dim': AE_LATENT,
            'best_val_mse': best_val}, MODEL_DIR / 'autoencoder.pt')
print(f'💾 Saved: models/autoencoder.pt')

---
## 📊 CELL 6 — Autoencoder Anomaly Scoring & Evaluation

In [ ]:
# ============================================================
# IDRS Part 4 — Autoencoder Scoring & Threshold Calibration
# ============================================================

def ae_score_dataset(ae, X: np.ndarray, batch_size: int = 1024) -> np.ndarray:
    """Compute per-sample reconstruction MSE for an array."""
    ae.eval()
    scores = []
    with torch.no_grad():
        for i in range(0, len(X), batch_size):
            xb = torch.tensor(X[i:i+batch_size],
                               dtype=torch.float32).to(DEVICE)
            xhat, _ = ae(xb)
            mse = ((xb - xhat)**2).mean(dim=-1).cpu().numpy()
            scores.append(mse)
    return np.concatenate(scores)


# ── Score all splits ──────────────────────────────────────
print('📊 Computing reconstruction errors...')
ae_scores_val  = ae_score_dataset(ae_model, X_ad_val)
ae_scores_test = ae_score_dataset(ae_model, X_ad_test)

# Normalise scores to [0,1] using val set statistics
ae_p01   = np.percentile(ae_scores_val, 1)
ae_p99   = np.percentile(ae_scores_val, 99)
ae_norm  = lambda s: np.clip((s - ae_p01) / (ae_p99 - ae_p01 + 1e-9), 0, 1)

ae_scores_val_n  = ae_norm(ae_scores_val)
ae_scores_test_n = ae_norm(ae_scores_test)

print(f'   Val  scores — mean: {ae_scores_val_n.mean():.4f}  '
      f'std: {ae_scores_val_n.std():.4f}')
print(f'   Test scores — mean: {ae_scores_test_n.mean():.4f}  '
      f'std: {ae_scores_test_n.std():.4f}')

# ── Evaluate on val (threshold calibration) ──────────────
ae_val_res  = evaluate_anomaly_detector(
    ae_scores_val_n, y_ad_val,
    'Autoencoder', 'Validation', 'ae', plot=True
)
AE_THRESHOLD = ae_val_res['opt_thresh']

# ── Evaluate on test ─────────────────────────────────────
ae_test_res = evaluate_anomaly_detector(
    ae_scores_test_n, y_ad_test,
    'Autoencoder', 'Test', 'ae', plot=True
)

# ── Per-class reconstruction error ───────────────────────
print('\n📊 Per-class reconstruction error (test set):')
print(f'  {"Class":<15} {"Mean MSE":>10} {"Std":>10} {"Detected%":>10}')
print('  ' + '-'*48)
for cid, cname in INT_TO_CLASS.items():
    mask   = (y_test_all == cid)
    if mask.sum() == 0:
        continue
    s      = ae_scores_test_n[mask]
    det_pct= (s >= AE_THRESHOLD).mean() * 100
    print(f'  {cname:<15} {s.mean():>10.4f} {s.std():>10.4f} {det_pct:>9.1f}%')

---
## 🌲 CELL 7 — Isolation Forest

In [ ]:
# ============================================================
# IDRS Part 4 — Isolation Forest
#
# Principle:
#   Anomalous samples are few and different — they are isolated
#   faster in a random partition tree (fewer splits needed).
#   Contamination = expected fraction of anomalies in the dataset.
#
# Trained on BENIGN-only data so it learns the normal manifold.
# Extended Isolation Forest variant (random hyperplanes) gives
# better detection on high-dimensional network feature vectors.
# ============================================================

print('🌲 Training Isolation Forest...')
t0 = time.time()

# Estimate contamination from val set class ratio
CONTAMINATION = float(np.clip(y_ad_val.mean(), 0.01, 0.49))
print(f'   Estimated contamination: {CONTAMINATION:.3f}')

iforest = IsolationForest(
    n_estimators  = 300,
    contamination = CONTAMINATION,
    max_samples   = min(len(X_ad_train), 10_000),  # subsample for speed
    max_features  = 0.8,        # random feature subsets (like random forest)
    bootstrap     = True,
    n_jobs        = -1,
    random_state  = SEED,
    warm_start    = False,
)
iforest.fit(X_ad_train)
print(f'   Trained in {time.time()-t0:.1f}s')

# ── Score: negate so higher = more anomalous ─────────────
# IsolationForest.decision_function: lower = more anomalous
if_scores_val  = -iforest.decision_function(X_ad_val)
if_scores_test = -iforest.decision_function(X_ad_test)

# Normalise
if_p01 = np.percentile(if_scores_val, 1)
if_p99 = np.percentile(if_scores_val, 99)
if_norm = lambda s: np.clip((s - if_p01)/(if_p99 - if_p01 + 1e-9), 0, 1)
if_scores_val_n  = if_norm(if_scores_val)
if_scores_test_n = if_norm(if_scores_test)

if_val_res  = evaluate_anomaly_detector(
    if_scores_val_n, y_ad_val,
    'IsolationForest', 'Validation', 'iforest', plot=True
)
IF_THRESHOLD = if_val_res['opt_thresh']

if_test_res = evaluate_anomaly_detector(
    if_scores_test_n, y_ad_test,
    'IsolationForest', 'Test', 'iforest', plot=True
)

joblib.dump(iforest, MODEL_DIR / 'isolation_forest.joblib')
print(f'💾 Saved: models/isolation_forest.joblib')

---
## ⭕ CELL 8 — One-Class SVM

In [ ]:
# ============================================================
# IDRS Part 4 — One-Class SVM
#
# Principle:
#   Learns a tight decision boundary around normal traffic in
#   a high-dimensional kernel space (RBF). Points outside the
#   boundary are flagged as anomalies.
#
# Note: One-Class SVM scales as O(n²) — we subsample the
# training set to keep training tractable.
# ============================================================

OC_SVM_SUBSAMPLE = min(8_000, len(X_ad_train))
rng = np.random.default_rng(SEED)
svm_train_idx = rng.choice(len(X_ad_train), OC_SVM_SUBSAMPLE, replace=False)
X_svm_train   = X_ad_train[svm_train_idx]

print(f'⭕ Training One-Class SVM (subsample={OC_SVM_SUBSAMPLE:,})...')
t0 = time.time()

ocsvm = OneClassSVM(
    kernel  = 'rbf',
    nu      = CONTAMINATION,     # upper bound on fraction of outliers
    gamma   = 'scale',
    shrinking= True,
    cache_size= 500,
)
ocsvm.fit(X_svm_train)
print(f'   Trained in {time.time()-t0:.1f}s')

# ── Scores ────────────────────────────────────────────────
# decision_function: negative = outside boundary = anomaly
svm_scores_val  = -ocsvm.decision_function(X_ad_val)
svm_scores_test = -ocsvm.decision_function(X_ad_test)

svm_p01 = np.percentile(svm_scores_val, 1)
svm_p99 = np.percentile(svm_scores_val, 99)
svm_norm = lambda s: np.clip((s - svm_p01)/(svm_p99 - svm_p01 + 1e-9), 0, 1)
svm_scores_val_n  = svm_norm(svm_scores_val)
svm_scores_test_n = svm_norm(svm_scores_test)

svm_val_res  = evaluate_anomaly_detector(
    svm_scores_val_n, y_ad_val,
    'OneClassSVM', 'Validation', 'ocsvm', plot=True
)
SVM_THRESHOLD = svm_val_res['opt_thresh']

svm_test_res = evaluate_anomaly_detector(
    svm_scores_test_n, y_ad_test,
    'OneClassSVM', 'Test', 'ocsvm', plot=True
)

joblib.dump(ocsvm, MODEL_DIR / 'one_class_svm.joblib')
print(f'💾 Saved: models/one_class_svm.joblib')

---
## 🔗 CELL 9 — Ensemble Anomaly Scorer

In [ ]:
# ============================================================
# IDRS Part 4 — Ensemble Anomaly Scorer
#
# Combines Autoencoder + Isolation Forest + One-Class SVM
# using learned weights optimised on the validation set.
#
# Weight optimisation: grid search over convex combinations
# maximising val ROC-AUC (subject to weights summing to 1).
# ============================================================

from itertools import product as iterproduct

print('🔗 Optimising ensemble weights on validation set...')

# Stack normalised scores: (n_val, 3)
val_stack  = np.column_stack([
    ae_scores_val_n, if_scores_val_n, svm_scores_val_n
])
test_stack = np.column_stack([
    ae_scores_test_n, if_scores_test_n, svm_scores_test_n
])

# Grid search over weights w = [w_ae, w_if, w_svm]
step         = 0.1
best_auc     = -np.inf
best_weights = np.array([1/3, 1/3, 1/3])

weight_range = np.arange(0, 1.01, step)
for w_ae, w_if in iterproduct(weight_range, weight_range):
    w_svm = 1.0 - w_ae - w_if
    if w_svm < 0 or w_svm > 1:
        continue
    w = np.array([w_ae, w_if, w_svm])
    ensemble_scores = val_stack @ w
    try:
        auc = roc_auc_score(y_ad_val, ensemble_scores)
        if auc > best_auc:
            best_auc     = auc
            best_weights = w.copy()
    except Exception:
        pass

W_AE, W_IF, W_SVM = best_weights
print(f'   Optimal weights — AE:{W_AE:.1f}  IF:{W_IF:.1f}  SVM:{W_SVM:.1f}')
print(f'   Val ROC-AUC    : {best_auc:.4f}')

# ── Ensemble scores ───────────────────────────────────────
ens_scores_val  = val_stack  @ best_weights
ens_scores_test = test_stack @ best_weights

ens_val_res  = evaluate_anomaly_detector(
    ens_scores_val, y_ad_val,
    'EnsembleAD', 'Validation', 'ensemble_ad', plot=True
)
ENS_THRESHOLD = ens_val_res['opt_thresh']

ens_test_res = evaluate_anomaly_detector(
    ens_scores_test, y_ad_test,
    'EnsembleAD', 'Test', 'ensemble_ad', plot=True
)

# ── Save ensemble config ──────────────────────────────────
ensemble_cfg = {
    'weights'       : {'AE': float(W_AE), 'IF': float(W_IF), 'SVM': float(W_SVM)},
    'thresholds'    : {
        'AE'       : float(AE_THRESHOLD),
        'IF'       : float(IF_THRESHOLD),
        'SVM'      : float(SVM_THRESHOLD),
        'Ensemble' : float(ENS_THRESHOLD),
    },
    'normalisation' : {
        'AE' : {'p01': float(ae_p01), 'p99': float(ae_p99)},
        'IF' : {'p01': float(if_p01), 'p99': float(if_p99)},
        'SVM': {'p01': float(svm_p01),'p99': float(svm_p99)},
    },
    'val_auc'       : float(best_auc),
    'test_results'  : ens_test_res,
}
with open(MODEL_DIR / 'anomaly_ensemble_cfg.json', 'w') as f:
    json.dump(ensemble_cfg, f, indent=2)
print(f'💾 Saved: models/anomaly_ensemble_cfg.json')

---
## 🧪 CELL 10 — Zero-Day Attack Simulation

In [ ]:
# ============================================================
# IDRS Part 4 — Zero-Day Attack Simulation
#
# Procedure:
#   1. Hold out ONE entire attack class from training
#      (simulating a novel unseen threat).
#   2. Train all anomaly detectors WITHOUT that class.
#   3. Measure detection rate of the held-out class at
#      the calibrated threshold.
#
# We repeat for each attack class and report detection rates.
# This tells us: "can our unsupervised detectors catch attacks
# they have never seen during training?"
# ============================================================

print('🧪 Zero-Day Attack Simulation')
print('   Holding out each attack class, testing detection rate...')
print('   (Using Isolation Forest for speed — same logic applies to AE)\n')

# We use the full (unbalanced) original arrays for this test
# so class counts are realistic
X_full   = np.vstack([X_train_all, X_val_all, X_test_all])
y_full   = np.concatenate([y_train_all, y_val_all, y_test_all])

ATTACK_CLASSES = [cid for cid in INT_TO_CLASS if cid != BENIGN_ID]
zeroday_results = []

for held_out_id in ATTACK_CLASSES:
    held_out_name = INT_TO_CLASS[held_out_id]

    # ── Build holdout-specific splits ─────────────────────
    # Train on benign only (zero-day: exclude held-out class too)
    train_mask   = (y_full == BENIGN_ID)
    X_zd_train   = X_full[train_mask]

    # Test: benign + held-out class ONLY
    test_mask    = np.isin(y_full, [BENIGN_ID, held_out_id])
    X_zd_test    = X_full[test_mask]
    y_zd_test    = (y_full[test_mask] != BENIGN_ID).astype(int)

    if y_zd_test.sum() < 10:
        print(f'   ⚠️  Skipping {held_out_name} — too few samples ({y_zd_test.sum()})')
        continue

    # ── Train Isolation Forest on benign only ─────────────
    n_sub    = min(len(X_zd_train), 5_000)
    sub_idx  = rng.choice(len(X_zd_train), n_sub, replace=False)

    zd_if    = IsolationForest(
        n_estimators  = 150,
        contamination = 0.05,
        max_samples   = min(n_sub, 4_000),
        n_jobs        = -1,
        random_state  = SEED,
    )
    zd_if.fit(X_zd_train[sub_idx])

    # ── Score test set ────────────────────────────────────
    zd_scores = -zd_if.decision_function(X_zd_test)

    # Calibrate threshold on a benign-only validation slice
    benign_scores = zd_scores[y_zd_test == 0]
    zd_thresh     = float(np.percentile(benign_scores, 95))  # 95th pctile of benign

    # ── Detection rate of held-out class ──────────────────
    attack_scores = zd_scores[y_zd_test == 1]
    det_rate      = float((attack_scores >= zd_thresh).mean())
    fpr           = float((zd_scores[y_zd_test == 0] >= zd_thresh).mean())

    try:
        auc = roc_auc_score(y_zd_test, zd_scores)
    except Exception:
        auc = float('nan')

    zeroday_results.append({
        'held_out_class': held_out_name,
        'n_attack_samples': int(y_zd_test.sum()),
        'detection_rate'  : round(det_rate, 4),
        'fpr'             : round(fpr,  4),
        'roc_auc'         : round(auc,  4),
        'threshold'       : round(zd_thresh, 6),
    })

    status = '🟢' if det_rate > 0.7 else ('🟡' if det_rate > 0.4 else '🔴')
    print(f'   {status} {held_out_name:<14}  '
          f'DetRate={det_rate:.3f}  FPR={fpr:.3f}  '
          f'AUC={auc:.3f}  n={y_zd_test.sum()}')

zd_df = pd.DataFrame(zeroday_results)

# ── Plot zero-day detection rates ────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

ax = axes[0]
colors = ['#2ecc71' if r>0.7 else ('#f39c12' if r>0.4 else '#e74c3c')
          for r in zd_df['detection_rate']]
bars = ax.barh(zd_df['held_out_class'], zd_df['detection_rate'],
               color=colors, edgecolor='white', linewidth=0.8)
ax.axvline(0.7, color='#27ae60', lw=2, ls='--', label='Good (0.70)')
ax.axvline(0.4, color='#f39c12', lw=2, ls='--', label='Fair (0.40)')
ax.set_xlabel('Zero-Day Detection Rate')
ax.set_title('Zero-Day Simulation — Detection Rate\n'
             '(Isolation Forest, novel class holdout)', fontweight='bold')
ax.set_xlim(0, 1.05)
ax.legend(fontsize=9)
for bar, v in zip(bars, zd_df['detection_rate']):
    ax.text(v+0.01, bar.get_y()+bar.get_height()/2,
            f'{v:.3f}', va='center', fontsize=8, fontweight='bold')

ax = axes[1]
ax.scatter(zd_df['fpr'], zd_df['detection_rate'],
           c=colors, s=120, zorder=5, edgecolors='black', linewidth=0.8)
for _, row in zd_df.iterrows():
    ax.annotate(row['held_out_class'],
                (row['fpr'], row['detection_rate']),
                textcoords='offset points', xytext=(6, 4), fontsize=8)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('Detection Rate (TPR)')
ax.set_title('Zero-Day: TPR vs FPR per Held-Out Class', fontweight='bold')
ax.axhline(0.7, color='#27ae60', lw=1.5, ls='--', alpha=0.6)
ax.set_xlim(-0.02, 0.5)
ax.set_ylim(-0.02, 1.05)

plt.suptitle('IDRS Zero-Day Attack Simulation\n'
             'Anomaly detector generalisation to novel unseen threats',
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'zeroday_simulation.png', bbox_inches='tight', dpi=150)
plt.show()

zd_df.to_csv(OUTPUT_DIR / 'zeroday_results.csv', index=False)
avg_det = zd_df['detection_rate'].mean()
print(f'\n   Average zero-day detection rate: {avg_det:.3f}')
print(f'💾 Saved: outputs/zeroday_simulation.png + zeroday_results.csv')

---
## 🌊 CELL 11 — BETH Dataset Integration

> **Dataset:** BETH (Behaviour-based Evaluation of Threat Hunting)  
> **Download:** https://www.kaggle.com/datasets/katehighnam/beth-dataset  
> **Place files in:** `data/beth/`

In [ ]:
# ============================================================
# IDRS Part 4 — BETH Dataset Host Behavioural Anomaly Detection
#
# BETH provides Linux host-level audit logs (syscalls, process
# events) from real honeypots. It complements CIC-IDS2017
# (network flows) with host-level behavioural signals.
#
# Features used:
#   processId, parentProcessId, userId, mountNamespace,
#   eventId, argsNum, returnValue, stackAddresses (count),
#   sus (suspicious flag), evil (ground truth)
# ============================================================

BETH_DIR = Path('data/beth')

BETH_FEATURE_COLS = [
    'processId', 'parentProcessId', 'userId', 'mountNamespace',
    'eventId', 'argsNum', 'returnValue', 'stackAddressesCount',
]
BETH_LABEL_COL = 'evil'

def load_beth(data_dir: Path,
              max_rows: int = 200_000) -> Optional[pd.DataFrame]:
    """Load BETH dataset CSV files."""
    csv_files = list(data_dir.glob('*.csv'))
    if not csv_files:
        return None
    frames = []
    for f in csv_files:
        try:
            chunk = pd.read_csv(f, nrows=max_rows, low_memory=False)
            frames.append(chunk)
            logger.info('BETH loaded: %s  rows=%d', f.name, len(chunk))
        except Exception as e:
            logger.warning('BETH load error: %s', e)
    return pd.concat(frames, ignore_index=True) if frames else None


def preprocess_beth(df: pd.DataFrame) -> Tuple[np.ndarray, np.ndarray]:
    """Preprocess BETH into feature matrix + binary labels."""
    available_feats = [c for c in BETH_FEATURE_COLS if c in df.columns]

    # stackAddresses is a list string in raw BETH — count list length
    if 'stackAddresses' in df.columns and 'stackAddressesCount' not in df.columns:
        df['stackAddressesCount'] = df['stackAddresses'].apply(
            lambda x: len(str(x).split(',')) if pd.notna(x) else 0
        )
        if 'stackAddressesCount' in BETH_FEATURE_COLS:
            available_feats.append('stackAddressesCount')

    X = df[available_feats].fillna(0).values.astype(np.float32)
    y = df[BETH_LABEL_COL].fillna(0).astype(int).values if BETH_LABEL_COL in df.columns \
        else np.zeros(len(df), dtype=int)
    return X, y, available_feats


beth_df = load_beth(BETH_DIR)

if beth_df is not None:
    print(f'✅ BETH dataset loaded: {len(beth_df):,} rows × {beth_df.shape[1]} cols')
    print(f'   Columns: {list(beth_df.columns)}')

    X_beth, y_beth, beth_feats = preprocess_beth(beth_df)
    print(f'   Feature matrix: {X_beth.shape}')
    print(f'   Anomaly rate:  {y_beth.mean()*100:.2f}%')
    print(f'   Features used: {beth_feats}')

    # ── Scale BETH features independently ─────────────────
    beth_scaler = StandardScaler()
    X_beth_tr_idx = np.where(y_beth == 0)[0][:10_000]
    beth_scaler.fit(X_beth[X_beth_tr_idx])
    X_beth_scaled = beth_scaler.transform(X_beth)

    # ── Train BETH-specific Isolation Forest ─────────────
    beth_if = IsolationForest(
        n_estimators=200, contamination=max(0.01, float(y_beth.mean())),
        n_jobs=-1, random_state=SEED
    )
    beth_if.fit(X_beth_scaled[X_beth_tr_idx])
    beth_scores = -beth_if.decision_function(X_beth_scaled)

    beth_res = evaluate_anomaly_detector(
        beth_scores, y_beth, 'BETH-IForest', 'Full', 'beth', plot=True
    )

    joblib.dump(beth_if,      MODEL_DIR / 'beth_iforest.joblib')
    joblib.dump(beth_scaler,  MODEL_DIR / 'beth_scaler.joblib')
    with open(MODEL_DIR / 'beth_meta.json', 'w') as fp:
        json.dump({'features': beth_feats,
                   'threshold': beth_res['opt_thresh']}, fp)
    BETH_AVAILABLE = True
    print(f'💾 BETH models saved.')

else:
    BETH_AVAILABLE = False
    print('ℹ️  BETH dataset not found in data/beth/')
    print('   Download from: https://www.kaggle.com/datasets/katehighnam/beth-dataset')
    print('   Place CSV files in data/beth/ and rerun this cell.')
    print('   Skipping BETH analysis — continuing with network-level anomaly detection.')

---
## 🌀 CELL 12 — Online Learning with River ADWIN

In [ ]:
# ============================================================
# IDRS Part 4 — Online Learning + Concept Drift Detection
#
# Educational platforms face CONCEPT DRIFT — traffic patterns
# evolve: new attack tools, new student behaviours, exam seasons.
#
# River framework provides:
#   1. ADWIN (ADaptive WINdowing): detects drift by monitoring
#      mean change in a sliding window. When drift is detected,
#      the online model resets its statistics.
#
#   2. Hoeffding Tree Classifier: incremental decision tree
#      that updates on each new sample without retraining.
#
# Simulation: stream test data in temporal order,
# inject an artificial concept drift at midpoint.
# ============================================================

from river import tree as river_tree
from river import drift as river_drift
from river import metrics as river_metrics
from river import compose, preprocessing as river_pre

print('🌀 Online Learning + ADWIN Concept Drift Detection')
print('   Simulating real-time traffic stream with injected drift...')

# ── Build stream data ─────────────────────────────────────
STREAM_SIZE    = min(10_000, len(X_test_all))
DRIFT_POINT    = STREAM_SIZE // 2

stream_X = X_test_all[:STREAM_SIZE]
stream_y = y_test_all[:STREAM_SIZE]

# Inject drift at midpoint: shift feature distributions
# Simulates: new attack tool generates different flow stats
drift_X = stream_X.copy()
drift_X[DRIFT_POINT:, :5] += 2.0   # shift first 5 features by 2σ

# ── River pipeline ────────────────────────────────────────
river_model = compose.Pipeline(
    river_pre.StandardScaler(),
    river_tree.HoeffdingTreeClassifier(
        grace_period     = 100,
        delta            = 1e-5,
        leaf_prediction  = 'mc',
        max_depth        = 20,
    )
)

# ADWIN drift detector: monitors binary error signal
adwin = river_drift.ADWIN(delta=0.002)

# Online metrics
online_accuracy = river_metrics.Accuracy()
online_f1       = river_metrics.MacroF1()
online_kappa    = river_metrics.CohenKappa()

# ── Streaming simulation ──────────────────────────────────
acc_history     = []
drift_detected  = []
drift_points    = []
reset_count     = 0

for i, (x_row, y_true) in enumerate(
    tqdm(zip(drift_X, stream_y), total=STREAM_SIZE, desc='Streaming')
):
    # Convert to River dict format
    x_dict = {f'f{j}': float(v) for j, v in enumerate(x_row)}

    # ── Predict ───────────────────────────────────────────
    y_pred = river_model.predict_one(x_dict)

    if y_pred is not None:
        # Update ADWIN with binary correct/incorrect signal
        is_correct = int(y_pred == y_true)
        adwin.update(is_correct)

        # Update online metrics
        online_accuracy.update(y_true, y_pred)
        online_f1.update(y_true, y_pred)
        online_kappa.update(y_true, y_pred)

        # Check for drift
        if adwin.drift_detected:
            drift_points.append(i)
            reset_count += 1

    # ── Learn ─────────────────────────────────────────────
    river_model.learn_one(x_dict, y_true)

    # Track rolling accuracy every 100 steps
    if i % 100 == 0:
        acc_history.append((i, float(online_accuracy.get())))

print(f'\n   Stream complete:')
print(f'   Final Accuracy : {online_accuracy.get():.4f}')
print(f'   Final F1 Macro : {online_f1.get():.4f}')
print(f'   Cohen Kappa    : {online_kappa.get():.4f}')
print(f'   Drift events   : {reset_count}  '
      f'(injected drift at index {DRIFT_POINT})')
print(f'   ADWIN alerts at: {drift_points[:10]}...' if len(drift_points)>10 else
      f'   ADWIN alerts at: {drift_points}')

# ── Plot online learning curve ────────────────────────────
fig, ax = plt.subplots(figsize=(14, 5))

steps   = [a[0] for a in acc_history]
accs    = [a[1] for a in acc_history]

ax.plot(steps, accs, lw=2, color='#3498db', label='Rolling Accuracy')
ax.fill_between(steps, accs, alpha=0.15, color='#3498db')
ax.axvline(DRIFT_POINT, color='#e74c3c', lw=2.5, ls='--',
           label=f'Injected Drift (idx={DRIFT_POINT})')

for dp in drift_points:
    ax.axvline(dp, color='#f39c12', lw=1.2, ls=':',
               alpha=0.6, label='ADWIN Alert' if dp == drift_points[0] else '')

ax.set_xlabel('Stream Index')
ax.set_ylabel('Rolling Accuracy')
ax.set_title(
    'Online Learning — Hoeffding Tree + ADWIN Concept Drift Detection\n'
    f'Final Accuracy={online_accuracy.get():.4f} | '
    f'ADWIN alerts={reset_count}',
    fontweight='bold'
)
ax.legend(fontsize=9)
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'online_learning_adwin.png', bbox_inches='tight', dpi=150)
plt.show()
print(f'💾 Saved: outputs/online_learning_adwin.png')

# Save River model state
import pickle
with open(MODEL_DIR / 'river_online_model.pkl', 'wb') as f:
    pickle.dump(river_model, f)
with open(MODEL_DIR / 'river_adwin.pkl', 'wb') as f:
    pickle.dump(adwin, f)
print(f'💾 Saved: models/river_online_model.pkl + river_adwin.pkl')

---
## 🏆 CELL 13 — Anomaly Detector Comparison Dashboard

In [ ]:
# ============================================================
# IDRS Part 4 — Anomaly Detector Comparison
# ============================================================

ad_results = [
    ae_test_res,
    if_test_res,
    svm_test_res,
    ens_test_res,
]
ad_df = pd.DataFrame(ad_results)
ad_df.to_csv(OUTPUT_DIR / 'anomaly_comparison.csv', index=False)

print('\n🏆 Anomaly Detector Comparison (Test Set)')
print('='*70)
print(ad_df[['detector','roc_auc','pr_auc','f1','tpr','fpr','fnr']].to_string(index=False))

best_ad = ad_df.loc[ad_df['roc_auc'].idxmax(), 'detector']
print(f'\n  🥇 Best anomaly detector (AUC): {best_ad}')

# ── Multi-metric grouped bar ──────────────────────────────
plot_metrics = ['roc_auc','pr_auc','f1','tpr']
fig, axes = plt.subplots(1, len(plot_metrics), figsize=(18, 5))

det_colors = ['#3498db','#e74c3c','#f39c12','#2ecc71']

for idx, metric in enumerate(plot_metrics):
    ax = axes[idx]
    vals = ad_df[metric].values
    bars = ax.bar(ad_df['detector'], vals,
                  color=det_colors[:len(ad_df)],
                  edgecolor='white', linewidth=0.8, width=0.6)
    best_bar_idx = int(np.argmax(vals))
    bars[best_bar_idx].set_edgecolor('#f1c40f')
    bars[best_bar_idx].set_linewidth(3)
    ax.set_title(metric.replace('_',' ').upper(), fontweight='bold', fontsize=9)
    ax.tick_params(axis='x', rotation=25, labelsize=8)
    ax.set_ylim(max(0, min(vals)-0.1), min(1.0, max(vals)+0.08))
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
                f'{v:.3f}', ha='center', va='bottom', fontsize=8, fontweight='bold')

plt.suptitle('Anomaly Detector Comparison — Test Set\n'
             '(Autoencoder · Isolation Forest · One-Class SVM · Ensemble)',
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'anomaly_comparison.png', bbox_inches='tight', dpi=150)
plt.show()
print(f'💾 Saved: outputs/anomaly_comparison.png')

---
## 🔁 CELL 14 — ROC Overlay: All Anomaly Detectors

In [ ]:
# ============================================================
# IDRS Part 4 — ROC Overlay: All Anomaly Detectors
# ============================================================

score_sets = [
    (ae_scores_test_n,  'Autoencoder', '#3498db'),
    (if_scores_test_n,  'Isolation Forest', '#e74c3c'),
    (svm_scores_test_n, 'One-Class SVM', '#f39c12'),
    (ens_scores_test,   'Ensemble', '#2ecc71'),
]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# ROC
ax = axes[0]
for scores, name, color in score_sets:
    fpr, tpr, _ = roc_curve(y_ad_test, scores)
    auc = roc_auc_score(y_ad_test, scores)
    ax.plot(fpr, tpr, lw=2.5 if 'Ensemble' in name else 1.8,
            color=color, label=f'{name} (AUC={auc:.4f})',
            ls='-' if 'Ensemble' in name else '--')
ax.plot([0,1],[0,1],'k--',lw=1,alpha=0.4,label='Random')
ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
ax.set_title('ROC Curves — All Anomaly Detectors', fontweight='bold')
ax.legend(fontsize=9)

# PR
ax = axes[1]
for scores, name, color in score_sets:
    prec, rec, _ = precision_recall_curve(y_ad_test, scores)
    ap = average_precision_score(y_ad_test, scores)
    ax.plot(rec, prec, lw=2.5 if 'Ensemble' in name else 1.8,
            color=color, label=f'{name} (AP={ap:.4f})',
            ls='-' if 'Ensemble' in name else '--')
ax.axhline(y_ad_test.mean(), color='k', ls=':', lw=1.2, alpha=0.5,
           label=f'Baseline={y_ad_test.mean():.3f}')
ax.set_xlabel('Recall'); ax.set_ylabel('Precision')
ax.set_title('PR Curves — All Anomaly Detectors', fontweight='bold')
ax.legend(fontsize=9)

plt.suptitle('IDRS Anomaly Detector Suite — Test Set',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'anomaly_roc_pr_overlay.png', bbox_inches='tight', dpi=150)
plt.show()
print(f'💾 Saved: outputs/anomaly_roc_pr_overlay.png')

---
## 💾 CELL 15 — Save All Artifacts & Part 4 Summary

In [ ]:
# ============================================================
# IDRS Part 4 — Save Artifacts & Summary Dashboard
# ============================================================

# ── Anomaly model registry ────────────────────────────────
anomaly_registry = {
    'Autoencoder' : {
        'checkpoint' : str(MODEL_DIR / 'autoencoder.pt'),
        'type'       : 'deep',
        'test_auc'   : ae_test_res['roc_auc'],
        'test_f1'    : ae_test_res['f1'],
        'threshold'  : float(AE_THRESHOLD),
        'norm'       : {'p01': float(ae_p01), 'p99': float(ae_p99)},
    },
    'IsolationForest' : {
        'checkpoint' : str(MODEL_DIR / 'isolation_forest.joblib'),
        'type'       : 'sklearn',
        'test_auc'   : if_test_res['roc_auc'],
        'test_f1'    : if_test_res['f1'],
        'threshold'  : float(IF_THRESHOLD),
        'norm'       : {'p01': float(if_p01), 'p99': float(if_p99)},
    },
    'OneClassSVM' : {
        'checkpoint' : str(MODEL_DIR / 'one_class_svm.joblib'),
        'type'       : 'sklearn',
        'test_auc'   : svm_test_res['roc_auc'],
        'test_f1'    : svm_test_res['f1'],
        'threshold'  : float(SVM_THRESHOLD),
        'norm'       : {'p01': float(svm_p01), 'p99': float(svm_p99)},
    },
    'Ensemble' : {
        'config'     : str(MODEL_DIR / 'anomaly_ensemble_cfg.json'),
        'weights'    : {'AE': float(W_AE), 'IF': float(W_IF), 'SVM': float(W_SVM)},
        'test_auc'   : ens_test_res['roc_auc'],
        'test_f1'    : ens_test_res['f1'],
        'threshold'  : float(ENS_THRESHOLD),
    },
    'OnlineLearner': {
        'checkpoint' : str(MODEL_DIR / 'river_online_model.pkl'),
        'adwin'      : str(MODEL_DIR / 'river_adwin.pkl'),
        'type'       : 'river',
        'final_acc'  : float(online_accuracy.get()),
        'final_f1'   : float(online_f1.get()),
        'drift_events': reset_count,
    },
    'BETH'         : {'available': BETH_AVAILABLE},
    'best_detector': best_ad,
    'zeroday': {
        'avg_detection_rate': float(avg_det),
        'per_class': zeroday_results,
    },
    'n_features'   : N_FEATURES,
    'benign_id'    : int(BENIGN_ID),
}

with open(MODEL_DIR / 'anomaly_registry.json', 'w') as f:
    json.dump(anomaly_registry, f, indent=2)

# ── Summary Dashboard ─────────────────────────────────────
fig = plt.figure(figsize=(20, 9))
fig.patch.set_facecolor('#0d1117')
ax  = fig.add_subplot(111)
ax.set_facecolor('#0d1117')
ax.axis('off')

ax.text(0.5, 0.97, '🛡️  IDRS PART 4 — SUMMARY',
        transform=ax.transAxes, ha='center', va='top',
        fontsize=20, fontweight='bold', color='white')
ax.text(0.5, 0.90,
        'Anomaly Detection · Zero-Day Simulation · BETH Integration · ADWIN Online Learning',
        transform=ax.transAxes, ha='center', va='top',
        fontsize=11, color='#8b949e')

block_data = [
    ('Autoencoder',     ae_test_res,  '#3498db', 0.02),
    ('Isolation Forest',if_test_res,  '#e74c3c', 0.27),
    ('One-Class SVM',   svm_test_res, '#f39c12', 0.52),
    ('Ensemble ⭐',     ens_test_res, '#2ecc71', 0.76),
]
for lbl, res, col, xoff in block_data:
    ax.text(xoff+0.10, 0.80, lbl, transform=ax.transAxes, ha='center',
            fontsize=10, fontweight='bold', color=col)
    lines = [
        f"AUC: {res['roc_auc']:.4f}",
        f"PR : {res['pr_auc']:.4f}",
        f"F1 : {res['f1']:.4f}",
        f"TPR: {res['tpr']:.4f}",
        f"FPR: {res['fpr']:.4f}",
        f"FNR: {res['fnr']:.4f}",
    ]
    for i, line in enumerate(lines):
        ax.text(xoff+0.10, 0.72-i*0.065, line, transform=ax.transAxes,
                ha='center', fontsize=8, color='white', fontfamily='monospace')

summary_items = [
    ('Zero-Day Avg Det.', f'{avg_det:.3f}'),
    ('ADWIN Drift Alerts', str(reset_count)),
    ('Online Accuracy',   f'{online_accuracy.get():.4f}'),
    ('Online F1',         f'{online_f1.get():.4f}'),
    ('BETH',              '✅ integrated' if BETH_AVAILABLE else 'not available'),
    ('Best Detector',     best_ad),
    ('Ensemble weights',  f'AE:{W_AE:.1f} IF:{W_IF:.1f} SVM:{W_SVM:.1f}'),
]
for i, (k, v) in enumerate(summary_items):
    c = 0.05 if i < 4 else 0.55
    r = i % 4
    y = 0.32 - r * 0.065
    ax.text(c,      y, f'{k}:', transform=ax.transAxes,
            fontsize=9, color='#8b949e', fontweight='bold')
    ax.text(c+0.17, y, v, transform=ax.transAxes,
            fontsize=9, color='#f0f6fc')

ax.text(0.5, 0.03,
        '→ Part 5: LLM-Powered Web Threat Analyser (DistilBERT fine-tune · SQLi · XSS · CSRF)',
        transform=ax.transAxes, ha='center', va='bottom',
        fontsize=9, color='#f39c12', style='italic')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'part4_summary.png', bbox_inches='tight',
            dpi=150, facecolor='#0d1117')
plt.show()

print('\n' + '='*60)
print('✅ PART 4 COMPLETE')
print(f'   Best anomaly detector: {best_ad}')
print(f'   Zero-day avg detection: {avg_det:.3f}')
print(f'   ADWIN drift alerts    : {reset_count}')
print(f'   → Proceed to IDRS_Part5_LLM_WebThreats.ipynb')
print('='*60)

---
## ✅ Part 4 — Completion Checklist

| Task | Status |
|---|---|
| Benign-only anomaly detection split | ✅ |
| Anomaly evaluation toolkit (ROC · PR · Youden threshold · score distribution) | ✅ |
| Deep Autoencoder with skip connections (U-Net style) | ✅ |
| AE trained on benign-only + cosine LR + early stopping | ✅ |
| Per-class reconstruction error analysis | ✅ |
| Isolation Forest (300 estimators, bootstrap, 80% features) | ✅ |
| One-Class SVM (RBF kernel, subsampled) | ✅ |
| Ensemble scorer with grid-search weight optimisation | ✅ |
| Zero-day simulation — novel class holdout for all attack types | ✅ |
| BETH dataset loader + host-level anomaly detection | ✅ |
| River Hoeffding Tree + ADWIN concept drift detection | ✅ |
| Online learning streaming simulation with injected drift | ✅ |
| ROC overlay of all four detectors | ✅ |
| Anomaly registry JSON + all models serialised | ✅ |

---
### ➡️ Next: `IDRS_Part5_LLM_WebThreats.ipynb`
- **DistilBERT fine-tuning** on SQL injection / XSS / command injection payloads
- HuggingFace free-tier Mistral for human-readable alert explanation
- Adversarial payload stress tests
- Per-class F1 + confusion matrix for web-specific threat classifier